# 04 · Fairness & Evaluation
Nationality fairness + per-language classifier fairness + honest failure cases + live spot-check.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
from src import config
from src.utils import load_json
fair=load_json(config.METRICS_DIR/'fairness_audit.json')
display(pd.DataFrame(fair['by_group']).T)
print('Spread (max-min mean churn prob):', fair['max_minus_min_mean_proba'], '->', fair['verdict'])

,n,mean_churn_proba,std_churn_proba
East_Asia,44.0,0.3573,0.4262
MENA,43.0,0.2800,0.4021
South_Asia,49.0,0.3524,0.4070
Southeast_Asia,44.0,0.3579,0.4365
Western,60.0,0.2691,0.3837


Spread (max-min mean churn prob): 0.0888 -> roughly even


## Per-language classifier fairness

In [2]:
pl=load_json(config.METRICS_DIR/'per_language_fairness.json')
pd.DataFrame(pl['per_language']).T

,language_name,n,accuracy,mean_confidence
ar,Arabic,12,1.0,0.7615
en,English,30,1.0,0.8239
hi,Hindi,6,1.0,0.7255
tl,Tagalog,15,1.0,0.6455


## Zero-shot per-language gap (ethics angle)

In [3]:
zs=load_json(config.METRICS_DIR/'zero_shot_comparison.json')
print(zs['zero_shot_accuracy_by_language']); print(zs['ethics_note'])

{'ar': 0.5556, 'en': 0.6364, 'hi': 0.0, 'tl': 0.5}
Zero-shot per-language accuracy is very uneven (see zero_shot_accuracy_by_language) — notably weak on Hindi — reinforcing the multilingual-fairness concern: an unguarded prompted LLM would deliver worse service in some languages.


## Documented failure cases (honest)

**Failure 1 — translation drops a clause / renames UAE.** opus-mt translated the Arabic leaver as *'I'm leaving the Emirates next month.'* (dropping the account-closure clause and rendering UAE as 'the Emirates'), so English regex missed the leaver intent. *Fix:* extract entities from BOTH the original and translated text and merge (`extract_entities_multi`), plus broaden the leaving pattern.

**Failure 2 — Guardrail hallucination catch.** A drafted goodbye proposed 'a special offer of AED 300'. Guardrails flagged the fabricated amount + incentive-on-goodbye and blocked auto-pass (`src/guardrails.py`).

In [4]:
from src.guardrails import check_draft
check_draft('We are sorry to see you go — here is a special offer of AED 300 to stay.', {'max_offer_value_aed':0.0,'dignified_goodbye':True})

{'guardrail_passed': False,
 'guardrail_warnings': ['Draft mentions AED 300 which does not match approved max offer value AED 0.',
  "Retention-incentive language '\\bspecial offer\\b' present on a Dignified Goodbye — must be a pressure-free farewell.",
  'Dignified Goodbye draft mentions a monetary amount — remove it.']}

## Live spot-check on the 3 pinned demo messages

In [5]:
from src.data_loader import get_message_row
from src.pipeline import run_pipeline
for mid in config.DEMO_MESSAGE_IDS:
    m=get_message_row(mid); r=run_pipeline(m['text'],m['customer_id'],message_id=mid,use_llm=False)
    d=r['decision']; print(mid,'|',d['action'],'| dignified_goodbye=',d['dignified_goodbye'])

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


M0010 | Dignified Goodbye Pathway | dignified_goodbye= True
M0007 | Dignified Goodbye Pathway | dignified_goodbye= True


M0243 | Retention Offer with Economics Check | dignified_goodbye= False
